[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/052.%20The%20EOQ%20with%20Quantity%20Discounts%20Problem/P52-Tier-4_executed.ipynb)

---


# 52. The EOQ with Quantity Discounts Problem

## Tier 4: Deep Learning Augmentation Method

### Purpose and scope

This notebook trains a multilayer perceptron (MLP) to predict the order quantity that minimizes annual cost in an EOQ problem with all-units quantity discounts. The training labels are generated by solving each synthetic scenario with the mathematical EOQ method implemented below. The learned model is therefore a supervised approximation to that mathematical solution—not a proof of optimality and not a replacement for the cost model.

The notebook proceeds in this order: generate synthetic scenarios and exact labels; scale the data and train an MLP; evaluate predictions on a held-out test set; examine selected scenarios; and measure prediction error and execution time. The mathematical solver is retained as the reference for labels and evaluation.

### Background: EOQ with all-units quantity discounts

Let \(D\) be annual demand in units per year, \(S\) the fixed ordering cost in currency per order, \(h\) the annual holding-cost rate, \(Q\) the order quantity in units, and \(c_i\) the unit purchase cost in price tier \(i\). If an order quantity belongs to tier \(i\), the annual total cost is

\[
TC_i(Q) = \frac{DS}{Q} + \frac{h c_i Q}{2} + D c_i.
\]

The terms are, respectively, annual ordering cost, annual cycle-stock holding cost, and annual purchase cost. For a fixed tier, the continuous EOQ is

\[
Q_i^{\mathrm{EOQ}} = \sqrt{\frac{2DS}{h c_i}}.
\]

The discount schedule restricts which quantities can use each unit cost. This notebook represents each tier as a tuple \((L_i,U_i,c_i)\), where \(L_i\) and \(U_i\) are inclusive integer quantity bounds and the final upper bound may be infinity. Because order quantities are whole units, the mathematical solver checks the feasible integer quantities nearest the tier-specific continuous EOQ, as well as the tier boundaries. It then selects the feasible candidate with the lowest annual total cost across all tiers.

### Learning formulation

A supervised learning example consists of scenario features \(x\) and a target \(Q^*(x)\), where \(Q^*(x)\) is the quantity returned by the mathematical solver. The MLP learns a function \(f_\theta(x)\) by minimizing squared prediction error on the training set:

\[
\min_{\theta} \frac{1}{n_{\mathrm{train}}}\sum_{j=1}^{n_{\mathrm{train}}}
\left(f_\theta(x_j)-Q^*(x_j)\right)^2.
\]

Here, \(\theta\) denotes the network weights and biases, and \(n_{\mathrm{train}}\) is the number of training scenarios. The implementation scales the features and target, then uses `MLPRegressor` with the specified hidden layers and optimizer. The synthetic input representation includes demand, ordering cost, holding rate, tier count, tier thresholds, and unit prices; unused tier slots are padded with zeros. This fixed-length representation lets one model accept scenarios with different numbers of tiers, up to the five tiers generated here.

The learned prediction is an estimate. Prediction error in units is not the same as cost regret: a quantity can be close to the optimal quantity but cross a discount threshold, or be farther away while incurring a similar cost. The notebook therefore reports quantity errors and evaluates predicted quantities using the scenario's annual cost function.

### Modeling assumptions and limits

- Demand and ordering cost are constant over the planning horizon, and replenishment is instantaneous.
- Shortages, lead times, safety stock, capacity limits, supplier reliability, and time-varying prices are outside this model.
- The discount is an all-units discount: the selected tier's unit price applies to every unit purchased.
- Tier bounds are inclusive integer quantities; tiers are ordered, non-overlapping, and cover all nonnegative integer quantities.
- Synthetic scenarios define the training distribution. Results on scenarios outside that distribution are not evidence of reliable generalization.
- The feature vector records the full tier schedule for up to five tiers. The network still approximates a discontinuous, tier-dependent optimization mapping, so its predictions must be checked against the mathematical cost model before operational use.

### What to examine

- Whether the training loss decreases and how validation scores evolve.
- How quantity predictions compare with mathematical targets on held-out synthetic data.
- How predicted quantities perform in the selected example scenarios and across additional test scenarios.
- What the timing measurements mean for this specific implementation. The mathematical benchmark checks a small number of tiers, so its runtime should not be interpreted as a general scaling comparison between optimization and neural networks.

The notebook does not implement historical-data learning, transfer learning, real-time market adaptation, storage constraints, or comparisons with separate heuristic and metaheuristic implementations. Those capabilities cannot be inferred from a model trained on synthetic scenarios with the features defined here.

In [ ]:
from itertools import product
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Fixed seeds make synthetic data generation, splitting, and MLP initialization reproducible.
np.random.seed(42)
random.seed(42)

print("Required libraries imported successfully.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("Scikit-learn MLPRegressor is available.")

In [ ]:
class EOQDeepLearningSolver:
    """Train an MLP to approximate mathematical EOQ solutions with quantity discounts."""

    def __init__(self):
        self.scaler_features = StandardScaler()
        self.scaler_target = MinMaxScaler()
        self.model = None
        self.training_history = {}
        self.input_dim = None

    @staticmethod
    def _validate_tiers(discount_tiers):
        """Check that tier bounds define an ordered, contiguous integer quantity range."""
        if not discount_tiers:
            raise ValueError("At least one discount tier is required.")

        expected_minimum = 0
        for index, (min_qty, max_qty, unit_cost) in enumerate(discount_tiers):
            if min_qty != expected_minimum:
                raise ValueError("Tiers must start at zero and be contiguous.")
            if not math.isfinite(unit_cost) or unit_cost <= 0:
                raise ValueError("Every unit cost must be finite and positive.")
            if index < len(discount_tiers) - 1:
                if not isinstance(max_qty, (int, np.integer)) or max_qty < min_qty:
                    raise ValueError("Finite tier bounds must be ordered integers.")
                expected_minimum = max_qty + 1
            elif max_qty != float("inf"):
                raise ValueError("The final tier must have an infinite upper bound.")

    @staticmethod
    def _build_features(demand, ordering_cost, holding_rate, discount_tiers):
        """Encode a complete tier schedule in the fixed-length network input."""
        if len(discount_tiers) > 5:
            raise ValueError("The feature representation supports at most five tiers.")

        min_quantities = [tier[0] for tier in discount_tiers[1:]]
        unit_costs = [tier[2] for tier in discount_tiers]
        features = [demand, ordering_cost, holding_rate, len(discount_tiers)]
        features.extend(min_quantities)
        features.extend([0] * (4 - len(min_quantities)))
        features.extend(unit_costs)
        features.extend([0] * (5 - len(unit_costs)))
        return np.asarray(features, dtype=float)

    @staticmethod
    def _total_annual_cost(quantity, demand, ordering_cost, holding_rate, unit_cost):
        """Evaluate ordering, cycle-stock holding, and purchase costs for one tier."""
        if quantity <= 0:
            raise ValueError("Order quantity must be positive.")
        holding_cost = quantity * unit_cost * holding_rate / 2
        annual_ordering_cost = demand * ordering_cost / quantity
        purchase_cost = demand * unit_cost
        return holding_cost + annual_ordering_cost + purchase_cost

    def generate_training_data(self, num_samples=1000):
        """Generate synthetic scenarios and label each with the mathematical solution."""
        data = []

        for _ in range(num_samples):
            demand = np.random.uniform(500, 50000)
            ordering_cost = np.random.uniform(20, 500)
            holding_rate = np.random.uniform(0.10, 0.35)

            num_tiers = np.random.randint(2, 6)
            base_price = np.random.uniform(5, 100)
            min_quantities = [0]
            unit_costs = [base_price]

            for tier in range(1, num_tiers):
                min_quantities.append(min_quantities[-1] + np.random.randint(100, 2000))
                discount_pct = min(0.30, tier * np.random.uniform(0.02, 0.08))
                unit_costs.append(base_price * (1 - discount_pct))

            discount_tiers = []
            for tier in range(num_tiers):
                max_qty = (
                    float("inf")
                    if tier == num_tiers - 1
                    else min_quantities[tier + 1] - 1
                )
                discount_tiers.append(
                    (min_quantities[tier], max_qty, unit_costs[tier])
                )

            optimal_qty, _, _ = self._solve_mathematical_eoq(
                demand, ordering_cost, holding_rate, discount_tiers
            )
            features = self._build_features(
                demand, ordering_cost, holding_rate, discount_tiers
            )
            data.append(np.append(features, optimal_qty))

        return np.asarray(data)

    def _solve_mathematical_eoq(
        self, demand, ordering_cost, holding_rate, discount_tiers
    ):
        """Find the minimum-cost feasible integer quantity by checking every tier."""
        if demand <= 0 or ordering_cost <= 0 or holding_rate <= 0:
            raise ValueError("Demand, ordering cost, and holding rate must be positive.")
        self._validate_tiers(discount_tiers)

        best_cost = float("inf")
        best_qty = None
        best_tier = None

        for index, (min_qty, max_qty, unit_cost) in enumerate(discount_tiers):
            eoq = math.sqrt(
                (2 * demand * ordering_cost) / (holding_rate * unit_cost)
            )
            candidates = {min_qty}
            if max_qty != float("inf"):
                candidates.add(max_qty)

            # The cost is convex within a tier, so the best integer quantity is
            # among the feasible integers nearest the continuous EOQ or a boundary.
            for quantity in (math.floor(eoq), math.ceil(eoq)):
                if quantity >= min_qty and (
                    max_qty == float("inf") or quantity <= max_qty
                ):
                    candidates.add(quantity)

            for quantity in candidates:
                if quantity <= 0:
                    continue
                cost = self._total_annual_cost(
                    quantity, demand, ordering_cost, holding_rate, unit_cost
                )
                if cost < best_cost:
                    best_cost = cost
                    best_qty = quantity
                    best_tier = index + 1

        if best_qty is None:
            raise ValueError("No positive feasible order quantity was found.")
        return best_qty, best_cost, best_tier

    def prepare_data(self, data):
        """Split labeled scenarios and fit scalers using training data only."""
        X = data[:, :-1]
        y = data[:, -1]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        X_train_scaled = self.scaler_features.fit_transform(X_train)
        X_test_scaled = self.scaler_features.transform(X_test)
        y_train_scaled = self.scaler_target.fit_transform(
            y_train.reshape(-1, 1)
        ).ravel()
        y_test_scaled = self.scaler_target.transform(
            y_test.reshape(-1, 1)
        ).ravel()

        return (
            X_train_scaled,
            X_test_scaled,
            y_train_scaled,
            y_test_scaled,
            X_test,
            y_test,
        )

    def build_model(self, input_dim):
        """Create the MLP architecture used to learn the EOQ quantity mapping."""
        self.input_dim = input_dim
        self.model = MLPRegressor(
            hidden_layer_sizes=(128, 64, 32),
            activation="relu",
            solver="adam",
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=1000,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=42,
        )
        print(f"Neural network hidden layers: {self.model.hidden_layer_sizes}")
        print(f"Input features: {self.input_dim}")
        print(f"Trainable parameters: approximately {self._count_parameters()}")

    def _count_parameters(self):
        """Count weight and bias parameters for the configured dense layers."""
        if self.model is None or self.input_dim is None:
            raise ValueError("Build the model before counting its parameters.")
        layers = [self.input_dim] + list(self.model.hidden_layer_sizes) + [1]
        return sum(
            layers[index] * layers[index + 1] + layers[index + 1]
            for index in range(len(layers) - 1)
        )

    def train_model(self, X_train, y_train, X_test, y_test_raw):
        """Fit the MLP and evaluate predictions in the original quantity units."""
        if self.model is None:
            raise ValueError("Build the model before training it.")

        print("Training the neural network...")
        self.model.fit(X_train, y_train)
        self.training_history = {
            "loss_curve": self.model.loss_curve_,
            "validation_scores": self.model.validation_scores_,
        }
        print("Training completed.")
        print(f"Final training loss: {self.model.loss_curve_[-1]:.6f}")
        if self.model.validation_scores_:
            print(
                "Best validation score: "
                f"{max(self.model.validation_scores_):.6f}"
            )

        y_pred_scaled = self.model.predict(X_test)
        y_pred = self.scaler_target.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).ravel()
        mse = mean_squared_error(y_test_raw, y_pred)
        mae = mean_absolute_error(y_test_raw, y_pred)
        r2 = r2_score(y_test_raw, y_pred)
        mape = np.mean(np.abs((y_test_raw - y_pred) / y_test_raw)) * 100

        print("\nHeld-out test-set metrics:")
        print(f"Mean squared error: {mse:.2f} units²")
        print(f"Mean absolute error: {mae:.2f} units")
        print(f"R² score: {r2:.4f}")
        print(f"Mean absolute percentage error: {mape:.2f}%")
        return y_pred, mse, mae, r2, mape

    def predict_optimal_quantity(
        self, demand, ordering_cost, holding_rate, discount_tiers
    ):
        """Predict a positive order quantity for a new, supported tier schedule."""
        if self.model is None:
            raise ValueError("Train the model before requesting predictions.")
        self._validate_tiers(discount_tiers)
        features = self._build_features(
            demand, ordering_cost, holding_rate, discount_tiers
        )
        features_scaled = self.scaler_features.transform(features.reshape(1, -1))
        prediction_scaled = self.model.predict(features_scaled)
        prediction = self.scaler_target.inverse_transform(
            prediction_scaled.reshape(-1, 1)
        )[0, 0]
        return max(1, int(round(prediction)))

print("EOQDeepLearningSolver class is ready.")

In [ ]:
dl_solver = EOQDeepLearningSolver()

print("Generating synthetic training scenarios...")
training_data = dl_solver.generate_training_data(num_samples=2000)
print(f"Generated samples: {len(training_data)}")
print(f"Feature count: {training_data.shape[1] - 1}")
print(
    "Optimal-quantity target range: "
    f"{training_data[:, -1].min():.0f} to {training_data[:, -1].max():.0f} units"
)

(
    X_train,
    X_test,
    y_train,
    y_test,
    X_test_raw,
    y_test_raw,
) = dl_solver.prepare_data(training_data)
print(f"Training samples: {len(X_train)}; test samples: {len(X_test)}")

dl_solver.build_model(input_dim=X_train.shape[1])
y_pred, mse, mae, r2, mape = dl_solver.train_model(
    X_train, y_train, X_test, y_test_raw
)

### Interpreting the training and test metrics

The loss curve tracks the MLP's optimization objective on scaled training targets; it is not an annual inventory cost. The validation score is used internally by early stopping to monitor held-out validation data drawn from the training split. The reported test metrics are calculated after converting predictions back to order-quantity units and comparing them with mathematical-solver labels in the held-out test set.

Mean absolute error and mean squared error measure quantity error in different ways. \(R^2\) describes variation explained relative to predicting the test-set mean; it is not a guarantee of operational quality. Mean absolute percentage error is meaningful here because the generated optimal quantities are positive. None of these metrics alone measures the annual cost of using a prediction.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("MLP Training and Held-Out Test Performance", fontsize=16)

axes[0, 0].plot(
    dl_solver.training_history["loss_curve"],
    color="steelblue",
    linewidth=2,
    label="Training loss",
)
axes[0, 0].set_title("Training Loss")
axes[0, 0].set_xlabel("Iteration")
axes[0, 0].set_ylabel("Loss on scaled targets")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

validation_scores = dl_solver.training_history["validation_scores"]
if validation_scores:
    axes[0, 1].plot(
        validation_scores,
        color="darkgreen",
        linewidth=2,
        label="Validation score",
    )
    axes[0, 1].set_ylabel("Validation R² score")
    axes[0, 1].legend()
else:
    axes[0, 1].text(
        0.5,
        0.5,
        "Validation scores are not available.",
        ha="center",
        va="center",
        transform=axes[0, 1].transAxes,
    )
axes[0, 1].set_title("Validation Progress")
axes[0, 1].set_xlabel("Iteration")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].scatter(y_test_raw, y_pred, alpha=0.6, s=30, label="Test scenarios")
plot_limits = [min(y_test_raw.min(), y_pred.min()), max(y_test_raw.max(), y_pred.max())]
axes[1, 0].plot(
    plot_limits,
    plot_limits,
    "r--",
    linewidth=2,
    label="Perfect quantity prediction",
)
axes[1, 0].set_title("Predicted and Mathematical Quantities")
axes[1, 0].set_xlabel("Mathematical quantity (units)")
axes[1, 0].set_ylabel("Predicted quantity (units)")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

errors = y_test_raw - y_pred
axes[1, 1].hist(errors, bins=30, color="skyblue", edgecolor="black", alpha=0.8)
axes[1, 1].axvline(0, color="red", linestyle="--", linewidth=2, label="Zero error")
axes[1, 1].set_title("Held-Out Quantity Error Distribution")
axes[1, 1].set_xlabel("Mathematical quantity minus prediction (units)")
axes[1, 1].set_ylabel("Number of scenarios")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

fig.tight_layout()
plt.show()

relative_errors = np.abs(errors / y_test_raw)
print("\nHeld-out prediction summary")
print(f"Mean absolute error: {mae:.2f} units")
print(f"Within 10% of the mathematical quantity: {np.mean(relative_errors < 0.10) * 100:.1f}%")
print(f"Within 5% of the mathematical quantity: {np.mean(relative_errors < 0.05) * 100:.1f}%")

### Selected scenario checks

For each scenario, the network supplies a quantity prediction. The mathematical solver supplies the reference quantity and minimum cost. The predicted quantity is evaluated at the price tier in which that quantity actually falls; this makes the cost comparison sensitive to discount thresholds. Quantity error and annual-cost error answer different questions, so both are reported.

In [ ]:
example_scenarios = [
    {
        "name": "MegaMart distribution center",
        "demand": 12000,
        "ordering_cost": 50,
        "holding_rate": 0.20,
        "tiers": [(0, 999, 5.00), (1000, 2499, 4.85), (2500, float("inf"), 4.75)],
    },
    {
        "name": "Simple two-tier discount",
        "demand": 2400,
        "ordering_cost": 50,
        "holding_rate": 0.20,
        "tiers": [(0, 999, 5.00), (1000, float("inf"), 4.75)],
    },
    {
        "name": "Five-tier discount",
        "demand": 5000,
        "ordering_cost": 75,
        "holding_rate": 0.25,
        "tiers": [
            (0, 499, 12.00),
            (500, 999, 10.50),
            (1000, 1999, 9.25),
            (2000, 4999, 8.75),
            (5000, float("inf"), 8.00),
        ],
    },
]

example_results = []
for scenario in example_scenarios:
    demand = scenario["demand"]
    ordering_cost = scenario["ordering_cost"]
    holding_rate = scenario["holding_rate"]
    tiers = scenario["tiers"]
    math_qty, math_cost, math_tier = dl_solver._solve_mathematical_eoq(
        demand, ordering_cost, holding_rate, tiers
    )
    predicted_qty = dl_solver.predict_optimal_quantity(
        demand, ordering_cost, holding_rate, tiers
    )
    predicted_tier = next(
        index + 1
        for index, (minimum, maximum, _) in enumerate(tiers)
        if minimum <= predicted_qty <= maximum
    )
    predicted_unit_cost = tiers[predicted_tier - 1][2]
    predicted_cost = dl_solver._total_annual_cost(
        predicted_qty, demand, ordering_cost, holding_rate, predicted_unit_cost
    )
    quantity_error = abs(predicted_qty - math_qty) / math_qty * 100
    cost_regret = predicted_cost - math_cost

    example_results.append(
        {
            "Scenario": scenario["name"],
            "Mathematical quantity (units)": math_qty,
            "Predicted quantity (units)": predicted_qty,
            "Mathematical tier": math_tier,
            "Predicted tier": predicted_tier,
            "Quantity error (%)": quantity_error,
            "Minimum annual cost": math_cost,
            "Predicted annual cost": predicted_cost,
            "Cost regret": cost_regret,
        }
    )

example_results_df = pd.DataFrame(example_results)
print(example_results_df.to_string(index=False, float_format=lambda value: f"{value:,.2f}"))

### Scenario-level evaluation

The following scenarios compare the learned quantity with the mathematical reference. Predicted cost uses the unit price associated with the predicted quantity's tier. A nonnegative cost regret means that the prediction's annual cost is no lower than the reference minimum, up to numerical precision. The MLP is not constrained to return the mathematical optimum, so quantity or cost differences are expected possibilities rather than evidence of a coding failure.

In [ ]:
scenarios = [
    {
        "name": "Small business",
        "demand": 1200,
        "ordering_cost": 35,
        "holding_rate": 0.18,
        "tiers": [(0, 499, 8.50), (500, float("inf"), 7.75)],
    },
    {
        "name": "Medium enterprise",
        "demand": 8500,
        "ordering_cost": 65,
        "holding_rate": 0.22,
        "tiers": [(0, 999, 15.00), (1000, 2499, 13.50), (2500, float("inf"), 12.00)],
    },
    {
        "name": "Large corporation",
        "demand": 25000,
        "ordering_cost": 120,
        "holding_rate": 0.25,
        "tiers": [
            (0, 999, 25.00),
            (1000, 4999, 22.50),
            (5000, 9999, 20.00),
            (10000, float("inf"), 18.50),
        ],
    },
    {
        "name": "Complex multi-tier",
        "demand": 15000,
        "ordering_cost": 85,
        "holding_rate": 0.20,
        "tiers": [
            (0, 299, 18.00),
            (300, 799, 16.50),
            (800, 1999, 15.00),
            (2000, 4999, 13.75),
            (5000, float("inf"), 12.50),
        ],
    },
]

scenario_results = []
for scenario in scenarios:
    demand = scenario["demand"]
    ordering_cost = scenario["ordering_cost"]
    holding_rate = scenario["holding_rate"]
    tiers = scenario["tiers"]
    math_qty, math_cost, math_tier = dl_solver._solve_mathematical_eoq(
        demand, ordering_cost, holding_rate, tiers
    )
    predicted_qty = dl_solver.predict_optimal_quantity(
        demand, ordering_cost, holding_rate, tiers
    )
    predicted_tier = next(
        index + 1
        for index, (minimum, maximum, _) in enumerate(tiers)
        if minimum <= predicted_qty <= maximum
    )
    predicted_cost = dl_solver._total_annual_cost(
        predicted_qty,
        demand,
        ordering_cost,
        holding_rate,
        tiers[predicted_tier - 1][2],
    )
    scenario_results.append(
        {
            "Scenario": scenario["name"],
            "Mathematical quantity": math_qty,
            "Predicted quantity": predicted_qty,
            "Mathematical annual cost": math_cost,
            "Predicted annual cost": predicted_cost,
            "Quantity error (%)": abs(predicted_qty - math_qty) / math_qty * 100,
            "Cost regret (%)": max(0, predicted_cost - math_cost) / math_cost * 100,
        }
    )

results_df = pd.DataFrame(scenario_results)
print(results_df.to_string(index=False, float_format=lambda value: f"{value:,.2f}"))

plot_names = results_df["Scenario"].tolist()
positions = np.arange(len(plot_names))
bar_width = 0.36
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].bar(
    positions - bar_width / 2,
    results_df["Mathematical quantity"],
    bar_width,
    label="Mathematical reference",
    color="steelblue",
)
axes[0].bar(
    positions + bar_width / 2,
    results_df["Predicted quantity"],
    bar_width,
    label="MLP prediction",
    color="coral",
)
axes[0].set_title("Order Quantity by Scenario")
axes[0].set_ylabel("Order quantity (units)")
axes[0].set_xticks(positions)
axes[0].set_xticklabels(plot_names, rotation=30, ha="right")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(
    positions,
    results_df["Cost regret (%)"],
    color="darkorange",
    alpha=0.8,
    label="Cost regret",
)
axes[1].set_title("Predicted Annual-Cost Regret")
axes[1].set_ylabel("Regret relative to mathematical minimum (%)")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(plot_names, rotation=30, ha="right")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

### Timing and batch-size experiment

This experiment records elapsed wall-clock time for the tier-by-tier mathematical calculation and the trained model's single-scenario prediction. It also measures repeated scenario processing. The mathematical method evaluates a small, finite list of tiers, while the prediction includes feature construction and scaling overhead. Consequently, observed timings depend on the machine, software environment, and fixed number of tiers; they do not establish that the MLP is generally faster or more scalable. Training time is not included in prediction time.

In [ ]:
test_sizes = [10, 50, 100, 500, 1000, 5000]
math_times = []
mlp_times = []

def make_timing_scenario(size):
    demand = size * 10
    ordering_cost = 50 + size * 0.01
    holding_rate = 0.20
    tiers = [(0, 999, 10.0), (1000, 4999, 9.0), (5000, float("inf"), 8.5)]
    return demand, ordering_cost, holding_rate, tiers

print("Single-scenario timing (milliseconds)")
print("Demand scale\tMathematical\tMLP prediction")
for size in test_sizes:
    demand, ordering_cost, holding_rate, tiers = make_timing_scenario(size)

    start = time.perf_counter()
    dl_solver._solve_mathematical_eoq(demand, ordering_cost, holding_rate, tiers)
    math_elapsed = (time.perf_counter() - start) * 1000

    start = time.perf_counter()
    dl_solver.predict_optimal_quantity(demand, ordering_cost, holding_rate, tiers)
    mlp_elapsed = (time.perf_counter() - start) * 1000

    math_times.append(math_elapsed)
    mlp_times.append(mlp_elapsed)
    print(f"{size}\t\t{math_elapsed:.3f}\t\t{mlp_elapsed:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].plot(test_sizes, math_times, "o-", label="Mathematical calculation")
axes[0].plot(test_sizes, mlp_times, "s-", label="MLP prediction")
axes[0].set_title("Single-Scenario Wall-Clock Time")
axes[0].set_xlabel("Demand scale (units per year = 10 × scale)")
axes[0].set_ylabel("Elapsed time (milliseconds)")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

batch_sizes = [10, 50, 100, 500, 1000]
batch_math_times = []
batch_mlp_times = []

for batch_size in batch_sizes:
    batch_scenarios = []
    for _ in range(batch_size):
        demand = np.random.uniform(1000, 20000)
        ordering_cost = np.random.uniform(30, 150)
        holding_rate = np.random.uniform(0.15, 0.30)
        num_tiers = np.random.randint(2, 5)
        base_price = np.random.uniform(5, 50)
        tiers = []
        for index in range(num_tiers):
            min_qty = index * 1000
            max_qty = float("inf") if index == num_tiers - 1 else (index + 1) * 1000 - 1
            unit_cost = base_price * (1 - index * 0.05)
            tiers.append((min_qty, max_qty, unit_cost))
        batch_scenarios.append((demand, ordering_cost, holding_rate, tiers))

    start = time.perf_counter()
    for demand, ordering_cost, holding_rate, tiers in batch_scenarios:
        dl_solver._solve_mathematical_eoq(demand, ordering_cost, holding_rate, tiers)
    batch_math_times.append((time.perf_counter() - start) * 1000)

    start = time.perf_counter()
    for demand, ordering_cost, holding_rate, tiers in batch_scenarios:
        dl_solver.predict_optimal_quantity(demand, ordering_cost, holding_rate, tiers)
    batch_mlp_times.append((time.perf_counter() - start) * 1000)

axes[1].plot(batch_sizes, batch_math_times, "o-", label="Mathematical calculations")
axes[1].plot(batch_sizes, batch_mlp_times, "s-", label="MLP predictions")
axes[1].set_title("Batch Wall-Clock Time")
axes[1].set_xlabel("Number of scenarios")
axes[1].set_ylabel("Elapsed time (milliseconds)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print("\nBatch timing measurements")
for batch_size, math_elapsed, mlp_elapsed in zip(
    batch_sizes, batch_math_times, batch_mlp_times
):
    print(
        f"{batch_size} scenarios: mathematical {math_elapsed:.2f} ms; "
        f"MLP {mlp_elapsed:.2f} ms"
    )

## Final interpretation

This experiment demonstrates a supervised-learning workflow for approximating mathematical EOQ solutions: synthetic scenarios provide inputs, the tier-by-tier mathematical method provides labels, and an MLP is assessed on held-out examples and selected scenarios. Test metrics describe agreement with those labels under the generated data distribution. Scenario cost regret adds an operational interpretation by evaluating a predicted quantity with the discount tier and annual cost formula that actually apply to it.

The mathematical method remains the reference for optimality. A learned prediction has no optimality guarantee, and the synthetic-data results do not establish performance on real procurement data or unseen discount structures. Before using a prediction in a decision process, assess both its feasibility and its annual cost under the relevant business constraints. The model in this notebook does not represent capacity, lead time, shortages, reliability, or other constraints outside the stated EOQ assumptions.

In [ ]:
summary_rows = []
math_elapsed_values = []
mlp_elapsed_values = []

for scenario in scenarios:
    demand = scenario["demand"]
    ordering_cost = scenario["ordering_cost"]
    holding_rate = scenario["holding_rate"]
    tiers = scenario["tiers"]

    start = time.perf_counter()
    math_qty, math_cost, math_tier = dl_solver._solve_mathematical_eoq(
        demand, ordering_cost, holding_rate, tiers
    )
    math_elapsed_values.append(time.perf_counter() - start)

    start = time.perf_counter()
    predicted_qty = dl_solver.predict_optimal_quantity(
        demand, ordering_cost, holding_rate, tiers
    )
    mlp_elapsed_values.append(time.perf_counter() - start)

    predicted_tier = next(
        index + 1
        for index, (minimum, maximum, _) in enumerate(tiers)
        if minimum <= predicted_qty <= maximum
    )
    predicted_cost = dl_solver._total_annual_cost(
        predicted_qty,
        demand,
        ordering_cost,
        holding_rate,
        tiers[predicted_tier - 1][2],
    )
    summary_rows.append(
        {
            "Scenario": scenario["name"],
            "Reference quantity": math_qty,
            "Predicted quantity": predicted_qty,
            "Reference tier": math_tier,
            "Predicted tier": predicted_tier,
            "Quantity error (%)": abs(predicted_qty - math_qty) / math_qty * 100,
            "Annual-cost regret": max(0, predicted_cost - math_cost),
        }
    )

summary_df = pd.DataFrame(summary_rows)
print("Selected-scenario summary")
print(summary_df.to_string(index=False, float_format=lambda value: f"{value:,.2f}"))
print("\nScope of the timing measurements")
print(f"Scenarios measured: {len(scenarios)}")
print(f"Mean mathematical calculation time: {np.mean(math_elapsed_values) * 1000:.4f} ms")
print(f"Mean MLP prediction time: {np.mean(mlp_elapsed_values) * 1000:.4f} ms")
print("These timings cover only the selected scenarios and the current software environment.")